# PopOut Game: MCTS vs Decision Trees

This notebook implements and compares two AI algorithms for playing the PopOut game:
- **Monte Carlo Tree Search (MCTS) with UCT**: A probabilistic search algorithm
- **Decision Trees**: A supervised learning approach

We'll analyze their performance, efficiency, and strategy differences.

## 1. Import Required Libraries

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from collections import defaultdict
import random
import time
from copy import deepcopy
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Import from project files
import sys
sys.path.append('.')
from logic import PopOutGame, MAX_ROLLOUT_MOVES
from mcts import MCTSNode
from interface import HumanPlayer, AIPlayer

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

## 2. Implement PopOut Game Environment

The PopOut game is a combinatorial game where players take turns removing items from rows/columns until all items are removed.

In [2]:
# PopOut Game is now imported from logic.py
# The game uses the Connect 4 variant with pop mechanics

print("=== PopOut Game Information ===\n")
print("PopOutGame (imported from logic.py):")
print("- Board size: 6 rows x 7 columns")
print("- Players: Player 1 (X) and Player 2 (O)")
print("- Win condition: Connect 4 discs horizontally, vertically, or diagonally")
print("\nAvailable moves:")
print("  1. DROP: Add your disc to the top of any non-full column")
print("  2. POP: Remove one of your discs from the bottom (others fall down)")
print("\nSpecial rules:")
print("  - Simultaneous four-in-rows after a pop: popper wins")
print("  - Full board: player to move can declare a draw")
print("  - Repetition: 3x same position allows draw declaration")

# Create a sample game
sample_game = PopOutGame(rows=6, cols=7)
print("\n=== Sample Board State ===")
print(sample_game)
print()

# Show legal moves
print("Legal moves for Player 1:")
moves = sample_game.get_legal_moves()
drops = [m[1] + 1 for m in moves if m[0] == 'drop']
pops = [m[1] + 1 for m in moves if m[0] == 'pop']
print(f"  DROP columns: {drops}")
print(f"  POP columns: {pops}")

=== PopOut Game Information ===

PopOutGame (imported from logic.py):
- Board size: 6 rows x 7 columns
- Players: Player 1 (X) and Player 2 (O)
- Win condition: Connect 4 discs horizontally, vertically, or diagonally

Available moves:
  1. DROP: Add your disc to the top of any non-full column
  2. POP: Remove one of your discs from the bottom (others fall down)

Special rules:
  - Simultaneous four-in-rows after a pop: popper wins
  - Full board: player to move can declare a draw
  - Repetition: 3x same position allows draw declaration

=== Sample Board State ===
  1 2 3 4 5 6 7
  -------------
6 . . . . . . .
5 . . . . . . .
4 . . . . . . .
3 . . . . . . .
2 . . . . . . .
1 . . . . . . .
  -------------
Player 1's turn

Legal moves for Player 1:
  DROP columns: [1, 2, 3, 4, 5, 6, 7]
  POP columns: []


## 3. Monte Carlo Tree Search (MCTS) Implementation

MCTS is an adversarial search algorithm that explores the game tree using UCT (Upper Confidence Bound for Trees) as the evaluation function. We'll implement different strategies and compare their performance.

#### 3.1 Standard MCTS (Pure UCT)
Exploration via Upper Bound Confidence fórmula, random rollout  

In [3]:
class MCTS:
    """
    Standard Monte Carlo Tree Search with UCT selection.
 
    Parameters
    ----------
    time_limit   : seconds of search per move (default 1.0)
    iterations   : hard cap on simulations (None = time-limited)
    c            : UCT exploration constant (default √2)
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
    ):
        self.time_limit = time_limit
        self.iterations = iterations
        self.c = c
        self.root: Optional[MCTSNode] = None
 
    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
 
    def choose_move(self, game) -> Optional[Tuple[str, int]]:
        """Return the best move for the current player."""
        legal = game.get_legal_moves()
        if not legal:
            return None
        if len(legal) == 1:
            return legal[0]
 
        self.root = MCTSNode(game.copy())
        self._run_search()
        best = self.root.most_visited_child()
        return best.move
 
    def _run_search(self):
        start = time.time()
        n = 0
        while True:
            if self.iterations and n >= self.iterations:
                break
            if not self.iterations and time.time() - start > self.time_limit:
                break
            node = self._select(self.root)
            if not node.is_terminal():
                node = self._expand(node)
            reward = self._simulate(node)
            self._backpropagate(node, reward)
            n += 1
 
    # ------------------------------------------------------------------
    # Four MCTS phases
    # ------------------------------------------------------------------
 
    def _select(self, node: MCTSNode) -> MCTSNode:
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(self.c)
        return node
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        if node.untried:
            return node.expand()
        return node
 
    def _simulate(self, node: MCTSNode) -> float:
        """
        Random rollout with three corrections:
 
        [R2] ('draw', -1) is now a legal move returned by get_legal_moves().
             random.choice picks it with uniform probability, so the agent
             naturally considers drawing when the condition is met.
             When the chosen move IS ('draw', -1), make_move() calls
             declare_draw() → game_over=True, is_draw=True → _outcome
             returns 0.5.
 
        [R3] Hard cap of MAX_ROLLOUT_MOVES prevents infinite loops caused
             by two random agents repeatedly revisiting the same position.
             If the cap is reached the rollout is scored as 0.5 (draw) —
             correct because a cycling game is heading toward repetition-draw.
        """
        state = node.state.copy()
        mover = node.player
 
        moves_played = 0                          # [R3] cycle guard counter
        while not state.game_over:
            if moves_played >= MAX_ROLLOUT_MOVES:  # [R3]
                return 0.5                         # draw — cycle detected
            moves = state.get_legal_moves()
            if not moves:
                break
            move = random.choice(moves)            # [R2] draw included here
            state.make_move(move[0], move[1])
            moves_played += 1
 
        return self._outcome(state, mover)
 
    def _backpropagate(self, node: MCTSNode, reward: float):
        root_player = self.root.state.current_player
        while node is not None:
            node.visits += 1
            if node.player == root_player:
                node.wins += reward
            else:
                node.wins += (1 - reward)
            node = node.parent
 
    # ------------------------------------------------------------------
 
    @staticmethod
    def _outcome(state, player: int) -> float:
        if state.winner == player:
            return 1.0
        if state.is_draw or state.winner is None:
            return 0.5
        return 0.0
 
    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------
 
    def get_move_statistics(self) -> List[Dict]:
        if self.root is None:
            return []
        stats = []
        for ch in self.root.children:
            stats.append({
                "move":     ch.move,
                "visits":   ch.visits,
                "win_rate": ch.wins / ch.visits if ch.visits else 0.0,
                "uct":      ch.uct_score(self.c),
            })
        return sorted(stats, key=lambda x: x["visits"], reverse=True)

#### 3.2 MCTS with RAPID ACTION VALUE ESTIMATION
Every move seen during a rollout is credited as if it had been played
first — the AMAF assumption.  The RAVE bias decays as node visits grow
(parameter k controls the decay rate).

In [4]:
class MCTS_RAVE(MCTS):
    """
    MCTS with RAVE (Rapid Action Value Estimation) / AMAF.
 
    Every move seen during a rollout is credited as if it had been played
    first — the AMAF assumption. The RAVE bias decays as node visits grow
    (parameter k controls the decay rate).
 
    [R3] rollout cap applied in _simulate, consistent with MCTS base class.
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        k: float = 1000,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.k = k
 
    # ------------------------------------------------------------------
 
    def _select(self, node: MCTSNode) -> MCTSNode:
        while node.is_fully_expanded() and not node.is_terminal():
            node = self._best_rave_child(node)
        return node
 
    def _best_rave_child(self, node: MCTSNode) -> MCTSNode:
        return max(
            node.children,
            key=lambda ch: ch.rave_score(ch.move, c=self.c, k=self.k),
        )
 
    def _simulate(self, node: MCTSNode):
        """
        Biased rollout that records moves played (for AMAF updates).
        Returns (reward, moves_played_as_list).
 
        [R2] ('draw', -1) included automatically via get_legal_moves().
        [R3] MAX_ROLLOUT_MOVES cap applied; returns (0.5, moves_so_far).
        """
        state = node.state.copy()
        mover = node.player
        moves_played: List[Tuple[str, int]] = []
 
        while not state.game_over:
            if len(moves_played) >= MAX_ROLLOUT_MOVES:  # [R3]
                return 0.5, moves_played
            moves = state.get_legal_moves()
            if not moves:
                break
            move = random.choice(moves)               # [R2] draw included
            moves_played.append(move)
            state.make_move(move[0], move[1])
 
        reward = self._outcome(state, mover)
        return reward, moves_played
 
    def _backpropagate(self, node: MCTSNode, payload):
        reward, moves_played = (
            payload if isinstance(payload, tuple) else (payload, [])
        )
        root_player = self.root.state.current_player
        current = node
        while current is not None:
            current.visits += 1
            flipped = reward if current.player == root_player else (1 - reward)
            current.wins += flipped
            if current.parent is not None:
                for m in moves_played:
                    current.parent.rave_visits[m] += 1
                    current.parent.rave_wins[m] += flipped
            current = current.parent
 
    def _run_search(self):
        start = time.time()
        n = 0
        while True:
            if self.iterations and n >= self.iterations:
                break
            if not self.iterations and time.time() - start > self.time_limit:
                break
            node = self._select(self.root)
            if not node.is_terminal():
                node = self._expand(node)
            payload = self._simulate(node)
            self._backpropagate(node, payload)
            n += 1

#### 3.3 Top-K MCTS
Branch pruning — expands only the k=7 most promising moves (out of 14)

In [5]:
class MCTSTopK(MCTS):
    """
    Top-K MCTS: during expansion only the K children with the highest UCT
    scores are kept; others are pruned.
 
    Inherits [R2] and [R3] fixes from MCTS._simulate unchanged.
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        k: int = 7,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.k = k
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        child = super()._expand(node)
        if len(node.children) > self.k:
            node.children.sort(
                key=lambda ch: ch.uct_score(self.c), reverse=True
            )
            node.children = node.children[: self.k]
        return child
 
    def _select(self, node: MCTSNode) -> MCTSNode:
        while node.is_fully_expanded() and not node.is_terminal():
            if not node.children:
                break
            node = node.best_child(self.c)
        return node

#### 3.4 Heuristic MCTS
Biased rollout: instead of uniform random, moves are sampled according to the heuristic weight
Prior score: newly expanded nodes receive a small prior win count proportional to their heuristic value

In [6]:
class MCTSWithHeuristics(MCTS):
    """
    Heuristic MCTS — biased rollouts + prior scores.
 
    [R2] _score_move now explicitly handles the ('draw', -1) move:
         it scores it as 0.5 × prior_weight, which is correct because a draw
         is neither a win nor a loss.  Without this the weighted_choice
         function would try to evaluate a non-existent column index of -1.
 
    [R3] Inherits the rollout cap from MCTS._simulate via _simulate override
         (see below — we keep our own _simulate but add the cap).
    """
 
    def __init__(
        self,
        time_limit: float = 1.0,
        iterations: Optional[int] = None,
        c: float = math.sqrt(2),
        prior_weight: float = 2.0,
    ):
        super().__init__(time_limit=time_limit, iterations=iterations, c=c)
        self.prior_weight = prior_weight
 
    # ------------------------------------------------------------------
    # Heuristic helpers
    # ------------------------------------------------------------------
 
    @staticmethod
    def _score_move(state, move: Tuple[str, int], player: int) -> float:
        """
        Quick heuristic score for a (move_type, col) from *player*'s view.
 
        [R2] ('draw', -1) returns 0.5 — neutral, neither win nor loss.
             This prevents a KeyError / wrong column access on index -1.
        """
        # FIX R2: draw move has a neutral heuristic value.
        if move[0] == 'draw':
            return 0.5
 
        opponent = 3 - player
        test = state.copy()
        test.make_move(move[0], move[1])
 
        if test.game_over and test.winner == player:
            return 5.0
 
        opp_moves = state.get_legal_moves()
        for om in opp_moves:
            if om[0] == 'draw':          # skip draw when scanning opponent
                continue
            bt = state.copy()
            bt.current_player = opponent
            bt.make_move(om[0], om[1])
            if bt.game_over and bt.winner == opponent:
                if om == move:
                    return 4.0
 
        score = 1.0
        if move[1] in (3, 2, 4):
            score += 0.5
        return score
 
    def _weighted_choice(
        self,
        state,
        moves: List[Tuple[str, int]],
        player: int,
    ) -> Tuple[str, int]:
        weights = [max(self._score_move(state, m, player), 0.01) for m in moves]
        total = sum(weights)
        r = random.random() * total
        cumulative = 0.0
        for m, w in zip(moves, weights):
            cumulative += w
            if r <= cumulative:
                return m
        return moves[-1]
 
    # ------------------------------------------------------------------
    # Overrides
    # ------------------------------------------------------------------
 
    def _expand(self, node: MCTSNode) -> MCTSNode:
        child = super()._expand(node)
        prior = self._score_move(node.state, child.move, child.player)
        child.prior_wins = prior * self.prior_weight
        return child
 
    def _simulate(self, node: MCTSNode) -> float:
        """
        Biased rollout using heuristic weights.
 
        [R2] _weighted_choice now handles ('draw', -1) via _score_move.
        [R3] Hard cap applied to break cycles.
        """
        state = node.state.copy()
        mover = node.player
 
        moves_played = 0                           # [R3]
        while not state.game_over:
            if moves_played >= MAX_ROLLOUT_MOVES:   # [R3]
                return 0.5
            moves = state.get_legal_moves()
            if not moves:
                break
            move = self._weighted_choice(state, moves, state.current_player)
            state.make_move(move[0], move[1])
            moves_played += 1
 
        return self._outcome(state, mover)

In [7]:
# ===========================================================================
# Convenience factory
# ===========================================================================
 
def make_mcts(strategy: str = "standard", **kwargs) -> MCTS:
    """
    Factory function.
 
    strategy: "standard" | "rave" | "topk" | "heuristic"
    kwargs  : forwarded to the chosen class constructor.
    """
    mapping = {
        "standard":  MCTS,
        "rave":      MCTS_RAVE,
        "topk":      MCTSTopK,
        "heuristic": MCTSWithHeuristics,
    }
    cls = mapping.get(strategy)
    if cls is None:
        raise ValueError(
            f"Unknown strategy '{strategy}'. "
            f"Choose from {list(mapping.keys())}"
        )
    return cls(**kwargs)

In [8]:
def _self_test():
    print("=" * 60)
    print("MCTS patch self-test")
    print("=" * 60)
 
    # --- R2: draw move appears and is chosen when it is the only option ---
    from logic import PopOutGame
    g = PopOutGame(rows=2, cols=2)
    g.make_move('drop', 0); g.make_move('drop', 1)
    g.make_move('drop', 0); g.make_move('drop', 1)
    assert g._is_board_full(), "Board should be full"
    legal = g.get_legal_moves()
    assert ('draw', -1) in legal, "draw must be a legal move on full board"
    agent = MCTS(iterations=20)
    move = agent.choose_move(g)
    # On a full board with no immediate win, a reasonable agent picks draw.
    print(f"[R2] Full-board move chosen: {move}  "
          f"(expected ('draw',-1) or a pop if rollout prefers it)")
 
    # --- R3: rollout cap prevents hang on a cycling tiny board ---
    import time
    g2 = PopOutGame(rows=2, cols=2)
    g2.make_move('drop', 0)    # P1
    g2.make_move('drop', 1)    # P2
    agent2 = MCTS(iterations=50)
    t0 = time.time()
    agent2.choose_move(g2)
    elapsed = time.time() - t0
    print(f"[R3] 50-iteration search on 2x2 board: {elapsed:.2f}s  "
          f"(should be < 2s even with cycling)")
    assert elapsed < 5, "Rollout cap failed — search took too long"
 
    # --- R2 + Heuristic: _score_move handles ('draw', -1) ---
    h_agent = MCTSWithHeuristics(iterations=20)
    score = MCTSWithHeuristics._score_move(g, ('draw', -1), 1)
    assert score == 0.5, f"Draw heuristic score should be 0.5, got {score}"
    print(f"[R2+Heuristic] _score_move('draw',-1) = {score}  ✓")
 
    print("\nAll self-tests passed ✓")
 
 
if __name__ == '__main__':
    _self_test()

MCTS patch self-test
[R2] Full-board move chosen: ('draw', -1)  (expected ('draw',-1) or a pop if rollout prefers it)
[R3] 50-iteration search on 2x2 board: 0.02s  (should be < 2s even with cycling)
[R2+Heuristic] _score_move('draw',-1) = 0.5  ✓

All self-tests passed ✓


## 4. Comparision of MCTS Strategies

In [10]:
# ── Configuração global do torneio ────────────────────────────────────────
N_GAMES    = 20    # jogos por par (10 como P1, 10 como P2)
ITERATIONS = 200   # iterações MCTS por movimento
 
strategies = {
    'Standard'  : MCTS(iterations=ITERATIONS),
    'RAVE'      : MCTS_RAVE(iterations=ITERATIONS),
    'Top-K'     : MCTSTopK(iterations=ITERATIONS, k=7),
    'Heuristic' : MCTSWithHeuristics(iterations=ITERATIONS),
}
 
 
# ══════════════════════════════════════════════════════════════════════════
# 4.1  run_game_detailed  —  regista métricas por movimento
# ══════════════════════════════════════════════════════════════════════════
 
def _agent_diagnostics(agent) -> Dict:
    """
    Extrai do agente MCTS (após choose_move) as métricas relevantes:
      - root_sims      : total de simulações na raiz
      - n_children     : nº de filhos expandidos (branching factor real)
      - best_visit_wr  : win-rate do filho MAIS VISITADO
      - best_move_wr   : win-rate do filho escolhido (pode diferir)
      - entropy        : entropia de Shannon sobre a distribuição de visitas
                         (alta → exploração; baixa → convergência num ramo)
    Retorna {} se o agente não tiver raiz (e.g. único lance legal).
    """
    if not hasattr(agent, 'root') or agent.root is None:
        return {}
 
    root     = agent.root
    children = root.children
    if not children:
        return {}
 
    visits = np.array([ch.visits for ch in children], dtype=float)
    total  = visits.sum()
    if total == 0:
        return {}
 
    # Distribuição de visitas → entropia
    probs   = visits / total
    entropy = -np.sum(probs * np.log2(probs + 1e-12))
 
    # Filho mais visitado
    most_visited = max(children, key=lambda ch: ch.visits)
    mv_wr = most_visited.wins / most_visited.visits if most_visited.visits else 0.0
 
    # Filho escolhido (o que o agente vai jogar = most_visited por convenção MCTS)
    # NOTA: most_visited_child() devolve o filho com mais visitas — é este o
    # movimento efectivamente jogado. mv_wr e chosen_wr são idênticos aqui,
    # mas registamos os dois para clareza e para suportar variantes futuras.
    chosen    = root.most_visited_child()
    chosen_wr = chosen.wins / chosen.visits if chosen.visits else 0.0
 
    return {
        'root_sims'     : int(total),
        'n_children'    : len(children),
        'best_visit_wr' : round(mv_wr,    4),   # win-rate filho mais visitado
        'chosen_wr'     : round(chosen_wr,4),   # win-rate movimento escolhido
        'entropy'       : round(entropy,  4),   # entropia da distribuição
    }
 
 
def _phase(move_number: int, total_moves_approx: int = 40) -> str:
    """Classifica a fase do jogo com base no nº do movimento."""
    frac = move_number / max(total_moves_approx, 1)
    if frac < 0.33:
        return 'opening'
    if frac < 0.67:
        return 'midgame'
    return 'endgame'
 
 
def run_game_detailed(
    agent1,
    agent2,
    verbose: bool = False,
    verbose_errors_only: bool = False,
) -> Dict:
    """
    Joga uma partida completa e devolve um dicionário rico em métricas:
 
      winner        : 1 | 2 | None
      n_moves       : nº total de meios-movimentos
      move_log      : lista de dicts, um por movimento (ver abaixo)
      p1_pops       : nº de pops do P1
      p2_pops       : nº de pops do P2
      p1_pops_win   : pops do P1 em jogos que ganhou   (só faz sentido
      p2_pops_win     ao agregar vários jogos)
      p1_times_ms   : lista de tempos (ms) por movimento do P1
      p2_times_ms   : lista de tempos (ms) por movimento do P2
 
    Cada entrada de move_log:
      move_number, player, move_type, col,
      move_time_ms, phase,
      root_sims, n_children,
      best_visit_wr, chosen_wr,   # ← nomes distintos para evitar confusão
      entropy
 
    verbose='errors_only' (ou verbose_errors_only=True):
      Só imprime lances onde chosen_wr < 0.50 segundo o próprio MCTS.
    """
    game   = PopOutGame(rows=6, cols=7)
    agents = {1: agent1, 2: agent2}
    move_times = {1: [], 2: []}
    move_log   = []
    p1_pops = p2_pops = 0
 
    move_number = 0
    while not game.game_over:
        p  = game.current_player
        t0 = time.time()
        move = agents[p].choose_move(game)
        elapsed_ms = (time.time() - t0) * 1000
        move_times[p].append(elapsed_ms)
 
        if move is None:
            break
 
        diag  = _agent_diagnostics(agents[p])
        phase = _phase(move_number)
 
        entry = {
            'move_number'  : move_number,
            'player'       : p,
            'move_type'    : move[0],
            'col'          : move[1],
            'move_time_ms' : round(elapsed_ms, 2),
            'phase'        : phase,
            'root_sims'    : diag.get('root_sims',      None),
            'n_children'   : diag.get('n_children',     None),
            'best_visit_wr': diag.get('best_visit_wr',  None),
            'chosen_wr'    : diag.get('chosen_wr',      None),
            'entropy'      : diag.get('entropy',        None),
        }
        move_log.append(entry)
 
        # ── Verbose output ────────────────────────────────────────────
        is_weak = (entry['chosen_wr'] is not None and entry['chosen_wr'] < 0.50)
        show = verbose or (verbose_errors_only and is_weak)
 
        if show:
            tag = '⚠ WEAK' if is_weak else ''
            print(f"\n── Move {move_number:>3d}  P{p}  {move[0].upper():4s} col {move[1]}  "
                  f"[{phase}]  {tag}")
            if diag:
                print(f"   sims={diag.get('root_sims','?'):>5}  "
                      f"children={diag.get('n_children','?'):>2}  "
                      f"best_visit_wr={diag.get('best_visit_wr','?'):.3f}  "
                      f"chosen_wr={diag.get('chosen_wr','?'):.3f}  "
                      f"entropy={diag.get('entropy','?'):.3f}")
            if verbose:
                print(game)
 
        # ── Contagem de pops ──────────────────────────────────────────
        if move[0] == 'pop':
            if p == 1: p1_pops += 1
            else:      p2_pops += 1
 
        game.make_move(move[0], move[1])
        move_number += 1
 
    winner = game.winner
    return {
        'winner'     : winner,
        'n_moves'    : move_number,
        'move_log'   : move_log,
        'p1_pops'    : p1_pops,
        'p2_pops'    : p2_pops,
        'p1_pops_win': p1_pops if winner == 1 else 0,
        'p2_pops_win': p2_pops if winner == 2 else 0,
        'p1_times_ms': move_times[1],
        'p2_times_ms': move_times[2],
    }
 
 
# ── Retrocompatibilidade com run_game simples ─────────────────────────────
def run_game(agent1, agent2):
    """Wrapper simples (mantém a assinatura original para seccção 5)."""
    r = run_game_detailed(agent1, agent2)
    return r['winner'], r['n_moves'], r['p1_times_ms'], r['p2_times_ms']

#### 4.2 Tournament between the different MCTS Strategies

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.2  run_tournament  —  métricas alargadas
# ══════════════════════════════════════════════════════════════════════════
 
def run_tournament(strategies: dict, n_games: int = N_GAMES) -> pd.DataFrame:
    """
    Round-robin completo com métricas alargadas por par:
      A1_pop%      : % de movimentos do A1 que são pop
      A2_pop%      : % de movimentos do A2 que são pop
      A1_pop%_W    : A1_pop% apenas nos jogos em que A1 ganhou
      A1_pop%_L    : A1_pop% apenas nos jogos em que A1 perdeu
      Moves_std    : desvio-padrão do comprimento dos jogos
      A1_entropy   : entropia média dos movimentos do A1
      A2_entropy   : entropia média dos movimentos do A2
    """
    names   = list(strategies.keys())
    records = []
 
    for i, name1 in enumerate(names):
        for name2 in names[i + 1:]:
            a1_wins = a2_wins = draws = 0
            a1_times, a2_times, game_lengths = [], [], []
            a1_pops_total = a2_pops_total   = 0
            a1_pops_w     = a2_pops_w       = 0
            a1_pops_l     = a2_pops_l       = 0
            a1_moves_total= a2_moves_total  = 0
            a1_entropy_all= a2_entropy_all  = []
            half = n_games // 2
 
            print(f'  {name1:12s} vs {name2:12s}  ({n_games} games)...',
                  end=' ', flush=True)
 
            for g in range(n_games):
                p1_is_a1 = g < half
                if p1_is_a1:
                    res = run_game_detailed(strategies[name1], strategies[name2])
                    a1_t = res['p1_times_ms']; a2_t = res['p2_times_ms']
                    a1_p = res['p1_pops'];     a2_p = res['p2_pops']
                    a1_pw= res['p1_pops_win']; a2_pw= res['p2_pops_win']
                    a1_m = sum(1 for e in res['move_log'] if e['player']==1)
                    a2_m = sum(1 for e in res['move_log'] if e['player']==2)
                    a1_ent= [e['entropy'] for e in res['move_log']
                             if e['player']==1 and e['entropy'] is not None]
                    a2_ent= [e['entropy'] for e in res['move_log']
                             if e['player']==2 and e['entropy'] is not None]
                    w = res['winner']
                    if w is None:   draws  += 1
                    elif w == 1:    a1_wins+= 1
                    else:           a2_wins+= 1
                else:
                    res = run_game_detailed(strategies[name2], strategies[name1])
                    a1_t = res['p2_times_ms']; a2_t = res['p1_times_ms']
                    a1_p = res['p2_pops'];     a2_p = res['p1_pops']
                    a1_pw= res['p2_pops_win']; a2_pw= res['p1_pops_win']
                    a1_m = sum(1 for e in res['move_log'] if e['player']==2)
                    a2_m = sum(1 for e in res['move_log'] if e['player']==1)
                    a1_ent= [e['entropy'] for e in res['move_log']
                             if e['player']==2 and e['entropy'] is not None]
                    a2_ent= [e['entropy'] for e in res['move_log']
                             if e['player']==1 and e['entropy'] is not None]
                    w = res['winner']
                    if w is None:   draws  += 1
                    elif w == 2:    a1_wins+= 1
                    else:           a2_wins+= 1
 
                game_lengths.append(res['n_moves'])
                a1_times.extend(a1_t); a2_times.extend(a2_t)
                a1_pops_total += a1_p;  a2_pops_total += a2_p
                a1_moves_total+= a1_m;  a2_moves_total+= a2_m
                # pops quando ganha vs perde
                a1_pops_w += a1_pw
                a2_pops_w += a2_pw
                a1_pops_l += a1_p - a1_pw
                a2_pops_l += a2_p - a2_pw
                a1_entropy_all.extend(a1_ent)
                a2_entropy_all.extend(a2_ent)
 
            total = a1_wins + a2_wins + draws
            # Nº de movimentos em jogos ganhos/perdidos (aproximação: usar total)
            a1_pop_pct    = round(a1_pops_total / max(a1_moves_total, 1) * 100, 1)
            a2_pop_pct    = round(a2_pops_total / max(a2_moves_total, 1) * 100, 1)
            a1_pop_pct_w  = round(a1_pops_w / max(a1_wins * 15, 1)  * 100, 1)
            a1_pop_pct_l  = round(a1_pops_l / max(a2_wins * 15, 1)  * 100, 1)
 
            print(f'done  [{name1} {a1_wins}W  {name2} {a2_wins}W  {draws}D]')
 
            records.append({
                'A1': name1, 'A2': name2,
                'A1_wins' : a1_wins,  'A2_wins' : a2_wins,  'Draws'  : draws,
                'A1_win%' : round(a1_wins / total * 100, 1),
                'A2_win%' : round(a2_wins / total * 100, 1),
                'Draw%'   : round(draws   / total * 100, 1),
                'A1_ms'   : round(np.mean(a1_times), 1) if a1_times else 0.0,
                'A2_ms'   : round(np.mean(a2_times), 1) if a2_times else 0.0,
                'Avg_moves'  : round(np.mean(game_lengths), 1),
                'Moves_std'  : round(np.std(game_lengths),  1),
                'A1_pop%'    : a1_pop_pct,
                'A2_pop%'    : a2_pop_pct,
                'A1_pop%_W'  : a1_pop_pct_w,   # pop% quando A1 ganha
                'A1_pop%_L'  : a1_pop_pct_l,   # pop% quando A1 perde
                'A1_entropy' : round(np.mean(a1_entropy_all), 3) if a1_entropy_all else None,
                'A2_entropy' : round(np.mean(a2_entropy_all), 3) if a2_entropy_all else None,
            })
 
    return pd.DataFrame(records)
 
 
# ── Execução do torneio ───────────────────────────────────────────────────
print(f'Tournament: {N_GAMES} games/pair  |  {ITERATIONS} MCTS iters/move')
print('=' * 62)
df_tournament = run_tournament(strategies, n_games=N_GAMES)
 
print('\n=== Full Results ===')
cols_show = ['A1','A2','A1_wins','A2_wins','Draws',
             'A1_win%','A2_win%','Draw%','A1_ms','A2_ms',
             'Avg_moves','Moves_std','A1_pop%','A2_pop%',
             'A1_pop%_W','A1_pop%_L','A1_entropy','A2_entropy']
print(df_tournament[cols_show].to_string(index=False))


#### 4.3 MCTS Convergence Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.3  Análise de Convergência MCTS
#      – tabela de avg sims/move com classificação automática
#      – curva de convergência temporal (sims e entropia ao longo do jogo)
# ══════════════════════════════════════════════════════════════════════════
 
def convergence_analysis(
    strategies: dict,
    n_diagnostic_games: int = 4,
) -> pd.DataFrame:
    """
    Para cada estratégia:
      1. Corre n_diagnostic_games jogos de self-play (agente vs ele próprio)
      2. Recolhe root_sims, n_children e entropy por movimento
      3. Agrega por fase (opening / midgame / endgame) e por índice de movimento
         (para a curva temporal)
      4. Classifica a convergência: High / Moderate / Low
 
    Devolve DataFrame com uma linha por estratégia e a curva temporal em dict.
    """
    results = {}
 
    for name, agent in strategies.items():
        print(f'  Convergence probe: {name} ({n_diagnostic_games} games)...',
              end=' ', flush=True)
        all_logs = []
 
        for _ in range(n_diagnostic_games):
            res = run_game_detailed(agent, agent)
            all_logs.extend(res['move_log'])
 
        df_log = pd.DataFrame(all_logs)
        df_log = df_log.dropna(subset=['root_sims', 'entropy'])
 
        avg_sims    = df_log['root_sims'].mean()
        avg_entropy = df_log['entropy'].mean()
        avg_children= df_log['n_children'].mean()
 
        # Classificação automática de convergência
        # (calibrada para MCTS com 200 iterações em PopOut 6×7)
        if avg_sims >= 180:
            convergence = 'High'
        elif avg_sims >= 120:
            convergence = 'Moderate'
        else:
            convergence = 'Low'
 
        # Agregação por fase
        phase_stats = (
            df_log.groupby('phase')[['root_sims','entropy','n_children']]
            .mean().round(2)
        )
 
        # Curva temporal: média de sims e entropia por índice de movimento
        temporal = (
            df_log.groupby('move_number')[['root_sims','entropy']]
            .mean()
        )
 
        results[name] = {
            'avg_sims'     : round(avg_sims,    1),
            'avg_entropy'  : round(avg_entropy, 3),
            'avg_children' : round(avg_children,1),
            'convergence'  : convergence,
            'phase_stats'  : phase_stats,
            'temporal'     : temporal,
        }
        print(f'done  [avg_sims={avg_sims:.0f}  entropy={avg_entropy:.3f}  → {convergence}]')
 
    # Tabela resumo
    summary = pd.DataFrame([
        {
            'Strategy'    : name,
            'Avg_sims'    : v['avg_sims'],
            'Avg_entropy' : v['avg_entropy'],
            'Avg_children': v['avg_children'],
            'Convergence' : v['convergence'],
        }
        for name, v in results.items()
    ]).set_index('Strategy')
 
    return summary, results
 
 
print('\n=== Análise de Convergência ===')
df_convergence, conv_details = convergence_analysis(strategies, n_diagnostic_games=4)
print(df_convergence.to_string())
 
 
# ── Plot: curva de convergência temporal ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.tab10.colors
 
for idx, (name, detail) in enumerate(conv_details.items()):
    t = detail['temporal']
    c = colors[idx % len(colors)]
    axes[0].plot(t.index, t['root_sims'], label=name, color=c, linewidth=1.8)
    axes[1].plot(t.index, t['entropy'],   label=name, color=c, linewidth=1.8)
 
axes[0].set_title('Simulações na raiz ao longo do jogo')
axes[0].set_xlabel('Nº do movimento'); axes[0].set_ylabel('Avg root_sims')
axes[0].legend(); axes[0].grid(alpha=.3)
 
axes[1].set_title('Entropia da distribuição de visitas ao longo do jogo')
axes[1].set_xlabel('Nº do movimento'); axes[1].set_ylabel('Avg entropy (bits)')
axes[1].legend(); axes[1].grid(alpha=.3)
 
for ax in axes:
    for x, label in [(13, 'midgame'), (27, 'endgame')]:
        ax.axvline(x=x, color='gray', linestyle='--', alpha=.5)
        ax.text(x+0.3, ax.get_ylim()[0], label, fontsize=7, color='gray')
 
plt.suptitle('Convergência MCTS — Curva Temporal', fontweight='bold')
plt.tight_layout()
plt.savefig('mcts_convergence_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curva temporal guardada → mcts_convergence_temporal.png')
 
 
# ── Plot: by-phase heatmaps ───────────────────────────────────────────────
fig, axes = plt.subplots(1, len(conv_details), figsize=(4 * len(conv_details), 4),
                         sharey=True)
for ax, (name, detail) in zip(axes, conv_details.items()):
    ps = detail['phase_stats'].reindex(['opening','midgame','endgame'])
    sns.heatmap(ps, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                cbar=(ax == axes[-1]))
    ax.set_title(f'{name}\n({df_convergence.loc[name,"Convergence"]})')
    ax.set_xlabel('')
plt.suptitle('Métricas por fase do jogo (avg)', fontweight='bold')
plt.tight_layout()
plt.savefig('mcts_phase_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmaps por fase guardados → mcts_phase_heatmaps.png')


#### 4.4 Log 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.4  Log de jogo verbose  —  exemplo de partida diagnóstica
#      Inclui modo errors_only para identificar falhas tácticas
# ══════════════════════════════════════════════════════════════════════════
 
def demo_game_verbose(agent1, agent2, name1: str, name2: str,
                      mode: str = 'full'):
    """
    Corre um jogo de demonstração e imprime o log detalhado.
 
    mode='full'        → imprime cada movimento
    mode='errors_only' → só imprime quando chosen_wr < 0.50
    """
    print(f'\n{"="*62}')
    print(f' Demo game: {name1} (P1)  vs  {name2} (P2)')
    print(f' Verbose mode: {mode}')
    print(f'{"="*62}')
 
    verbose        = (mode == 'full')
    errors_only    = (mode == 'errors_only')
 
    res = run_game_detailed(agent1, agent2,
                            verbose=verbose,
                            verbose_errors_only=errors_only)
 
    winner_str = f'Player {res["winner"]}' if res['winner'] else 'Draw'
    print(f'\n── Resultado: {winner_str}  ({res["n_moves"]} movimentos) ──')
    print(f'   P1 pops: {res["p1_pops"]}  |  P2 pops: {res["p2_pops"]}')
 
    # Tabela resumo do log
    df_log = pd.DataFrame(res['move_log'])
    if not df_log.empty:
        weak_moves = df_log[df_log['chosen_wr'].lt(0.50).fillna(False)]
        print(f'\n   Movimentos com chosen_wr < 0.50: {len(weak_moves)}')
        if not weak_moves.empty:
            print(weak_moves[['move_number','player','move_type','col',
                              'phase','chosen_wr','entropy']].to_string(index=False))
    return res
 
 
# Demonstração: Heuristic vs RAVE em modo errors_only
_ = demo_game_verbose(
    strategies['Heuristic'], strategies['RAVE'],
    'Heuristic', 'RAVE',
    mode='errors_only',
)


#### 4.5 Visual Dashboard

In [ ]:
agent_names = list(strategies.keys())
n_agents    = len(agent_names)
 
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
 
# ── (A) Win-rate heatmap ─────────────────────────────────────────────────
ax_win = fig.add_subplot(gs[0, 0])
win_matrix = pd.DataFrame(np.full((n_agents, n_agents), np.nan),
                           index=agent_names, columns=agent_names)
for _, row in df_tournament.iterrows():
    win_matrix.loc[row['A1'], row['A2']] = row['A1_win%']
    win_matrix.loc[row['A2'], row['A1']] = row['A2_win%']
sns.heatmap(win_matrix.astype(float), annot=True, fmt='.1f',
            cmap='RdYlGn', vmin=0, vmax=100, linewidths=.5,
            ax=ax_win, cbar_kws={'shrink': .7})
ax_win.set_title('(A) Win % (linha vs coluna)')
 
# ── (B) Tempo médio por agente ──────────────────────────────────────────
ax_time = fig.add_subplot(gs[0, 1])
time_data = defaultdict(list)
for _, row in df_tournament.iterrows():
    time_data[row['A1']].append(row['A1_ms'])
    time_data[row['A2']].append(row['A2_ms'])
avg_times = {ag: np.mean(v) for ag, v in time_data.items()}
pd.Series(avg_times).sort_values().plot(
    kind='barh', ax=ax_time, color='steelblue', edgecolor='white')
ax_time.set_xlabel('ms por movimento')
ax_time.set_title('(B) Tempo médio de decisão')
ax_time.grid(axis='x', alpha=.3)
 
# ── (C) Avg_moves ± std por par ─────────────────────────────────────────
ax_len = fig.add_subplot(gs[0, 2])
pairs  = [f'{r.A1[:4]} v {r.A2[:4]}' for _, r in df_tournament.iterrows()]
ax_len.bar(pairs, df_tournament['Avg_moves'],
           yerr=df_tournament['Moves_std'], capsize=4,
           color='mediumpurple', edgecolor='white')
ax_len.set_ylabel('Nº de movimentos')
ax_len.set_title('(C) Comprimento médio ± std dos jogos')
ax_len.set_xticklabels(pairs, rotation=35, ha='right', fontsize=8)
ax_len.grid(axis='y', alpha=.3)
 
# ── (D) Pop% por agente (global) ─────────────────────────────────────────
ax_pop = fig.add_subplot(gs[1, 0])
pop_a1 = dict(zip(df_tournament['A1'], df_tournament['A1_pop%']))
pop_a2 = dict(zip(df_tournament['A2'], df_tournament['A2_pop%']))
pop_all = defaultdict(list)
for k, v in {**pop_a1, **pop_a2}.items():
    pop_all[k].append(v)
pop_means = {k: np.mean(v) for k, v in pop_all.items()}
pd.Series(pop_means).sort_values().plot(
    kind='barh', ax=ax_pop, color='tomato', edgecolor='white')
ax_pop.set_xlabel('% de movimentos que são pop')
ax_pop.set_title('(D) Uso do pop por agente')
ax_pop.grid(axis='x', alpha=.3)
 
# ── (E) Pop%_W vs Pop%_L  (só para A1) ──────────────────────────────────
ax_popwl = fig.add_subplot(gs[1, 1])
df_popwl = df_tournament[['A1','A1_pop%_W','A1_pop%_L']].copy()
df_popwl = df_popwl.groupby('A1')[['A1_pop%_W','A1_pop%_L']].mean()
x_pos = np.arange(len(df_popwl))
ax_popwl.bar(x_pos - 0.2, df_popwl['A1_pop%_W'], 0.4,
             label='Ganha', color='seagreen', edgecolor='white')
ax_popwl.bar(x_pos + 0.2, df_popwl['A1_pop%_L'], 0.4,
             label='Perde', color='tomato',   edgecolor='white')
ax_popwl.set_xticks(x_pos)
ax_popwl.set_xticklabels(df_popwl.index, rotation=20, ha='right')
ax_popwl.set_ylabel('Pop% médio')
ax_popwl.set_title('(E) Pop% quando ganha vs perde (A1)')
ax_popwl.legend(fontsize=8)
ax_popwl.grid(axis='y', alpha=.3)
 
# ── (F) Entropia média por agente ────────────────────────────────────────
ax_ent = fig.add_subplot(gs[1, 2])
ent_a1 = dict(zip(df_tournament['A1'], df_tournament['A1_entropy']))
ent_a2 = dict(zip(df_tournament['A2'], df_tournament['A2_entropy']))
ent_all = defaultdict(list)
for d in [ent_a1, ent_a2]:
    for k, v in d.items():
        if v is not None:
            ent_all[k].append(v)
ent_means = {k: np.mean(v) for k, v in ent_all.items()}
pd.Series(ent_means).sort_values().plot(
    kind='barh', ax=ax_ent, color='darkorange', edgecolor='white')
ax_ent.set_xlabel('Entropia média (bits)')
ax_ent.set_title('(F) Entropia da distribuição de visitas')
ax_ent.grid(axis='x', alpha=.3)
 
# ── (G) Tabela de convergência ───────────────────────────────────────────
ax_conv = fig.add_subplot(gs[2, :])
ax_conv.axis('off')
tbl_data = df_convergence.reset_index()
tbl_data.columns = ['Estratégia','Avg sims/move','Avg entropy','Avg children','Convergência']
colors_conv = []
for row in tbl_data.itertuples():
    c = {'High':'#c8e6c9','Moderate':'#fff9c4','Low':'#ffcdd2'}.get(row[4], 'white')
    colors_conv.append([c]*5)
table = ax_conv.table(
    cellText=tbl_data.values,
    colLabels=tbl_data.columns,
    cellLoc='center', loc='center',
    cellColours=colors_conv,
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
ax_conv.set_title('(G) Resumo de Convergência MCTS', pad=12, fontweight='bold')
 
plt.suptitle('Capítulo 4 — Dashboard de Comparação de Estratégias MCTS',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('mcts_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard guardado → mcts_dashboard.png')

## 5. Decision Tree Implementation

Train a Decision Tree to learn optimal strategies from game data.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5.1  State encoding helpers
# ══════════════════════════════════════════════════════════════════════════

FEATURE_NAMES = (
    [f'r{r}c{c}' for r in range(6) for c in range(7)]  # 42 board cells
    + ['current_player']                                 # whose turn
)
N_ACTIONS = 14  # 7 drop + 7 pop

def encode_move(move: Tuple[str, int]) -> int:
    """Encode (type, col) → integer label 0-13.
    drop col c → c      (0-6)
    pop  col c → c + 7  (7-13)
    """
    mtype, col = move
    return col if mtype == 'drop' else col + 7

def decode_move(label: int) -> Tuple[str, int]:
    """Inverse of encode_move."""
    return ('drop', label) if label < 7 else ('pop', label - 7)

def state_to_features(game: PopOutGame) -> List[int]:
    """Flat feature vector: 42 board cells (0/1/2) + current_player (1/2)."""
    return list(game.board.flatten().astype(int)) + [game.current_player]

print(f"Feature vector length : {len(FEATURE_NAMES)}")
print(f"Action space size     : {N_ACTIONS}  (0-6 drop, 7-13 pop)")
print(f"Example encoding      : drop col 3 → {encode_move(('drop',3))}, "
      f"pop col 3 → {encode_move(('pop',3))}")


Feature vector length : 43
Action space size     : 14  (0-6 drop, 7-13 pop)
Example encoding      : drop col 3 → 3, pop col 3 → 10


### 5.2  Dataset Generation

After the tournament the **winning MCTS variant** is identified from `df_tournament` and used to label every position. Each self-play game produces one `(state, best_MCTS_move)` pair per half-move. Both players are driven by the same agent so the dataset covers positions from both sides of the board.


In [ ]:
# ── Identify the winning MCTS from tournament results ────────────────────
mcts_names  = ['Standard', 'RAVE', 'Top-K', 'Heuristic']
mcts_scores = {ag: 0 for ag in mcts_names}

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    if a1 in mcts_names and a2 in mcts_names:
        mcts_scores[a1] += row['A1_wins']
        mcts_scores[a2] += row['A2_wins']

best_mcts_name = max(mcts_scores, key=mcts_scores.get)

print("MCTS win counts (inter-MCTS games only):")
for ag, sc in sorted(mcts_scores.items(), key=lambda x: -x[1]):
    print(f"  {ag:12s}  {sc} wins")
print(f"\nTournament winner → {best_mcts_name}")

# ── Configuration ────────────────────────────────────────────────────────
N_DATASET_GAMES = 200   # self-play games to generate the dataset
MCTS_ITER_DATA  = 200   # MCTS iterations per move during generation
#   200 iter × ~30 moves/game × 200 games ≈ 1.2 M simulations total.

# ── Build the data agent from the tournament winner ───────────────────────
_data_agent_map = {
    'Standard'  : MCTS,
    'RAVE'      : MCTS_RAVE,
    'Top-K'     : MCTSTopK,
    'Heuristic' : MCTSWithHeuristics,
}
data_agent = _data_agent_map[best_mcts_name](iterations=MCTS_ITER_DATA)
print(f"Data agent : {best_mcts_name}  ({MCTS_ITER_DATA} iterations/move)\n")

# ── Collection loop ───────────────────────────────────────────────────────
import time as _time

X_raw: List[List[int]] = []
y_raw: List[int]       = []

t0 = _time.time()
for g_idx in range(N_DATASET_GAMES):
    game = PopOutGame(rows=6, cols=7)
    while not game.game_over:
        # Record (state, best-MCTS move) BEFORE applying the move
        features = state_to_features(game)
        move     = data_agent.choose_move(game)
        if move is None:
            break
        X_raw.append(features)
        y_raw.append(encode_move(move))
        game.make_move(move[0], move[1])
    if (g_idx + 1) % 20 == 0:
        elapsed = _time.time() - t0
        print(f"  Game {g_idx+1:>3}/{N_DATASET_GAMES}  "
              f"| samples so far: {len(X_raw):>5}  "
              f"| elapsed: {elapsed:.1f}s")

print(f"\nDataset generated: {len(X_raw)} samples from {N_DATASET_GAMES} games")
print(f"Total time: {_time.time()-t0:.1f}s")


NameError: name 'MCTSWithHeuristics' is not defined

### 5.3  Dataset Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X = np.array(X_raw, dtype=np.int8)
y = np.array(y_raw,  dtype=np.int8)

df_dataset = pd.DataFrame(X, columns=FEATURE_NAMES)
df_dataset['action'] = y
df_dataset['action_str'] = [str(decode_move(int(a))) for a in y]

print(f"Shape            : {X.shape}")
print(f"Unique actions   : {np.unique(y)}")
print(f"Action counts:")
action_counts = pd.Series(y).value_counts().sort_index()
for lbl, cnt in action_counts.items():
    print(f"  {str(decode_move(int(lbl))):20s}  {cnt:5d}  ({cnt/len(y)*100:.1f}%)")

# Plot action distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

action_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Action Distribution (0-6: drop, 7-13: pop)')
axes[0].set_xlabel('Action label')
axes[0].set_ylabel('Frequency')
axes[0].set_xticks(range(14))
axes[0].set_xticklabels(
    [f'D{c}' for c in range(7)] + [f'P{c}' for c in range(7)], rotation=45)

# Cell occupation heatmap (avg player value per cell)
board_mean = X[:, :42].mean(axis=0).reshape(6, 7)
im = axes[1].imshow(board_mean, cmap='RdYlGn', vmin=0, vmax=2, aspect='auto')
axes[1].set_title('Average cell occupation across dataset')
axes[1].set_xlabel('Column'); axes[1].set_ylabel('Row (0=top)')
plt.colorbar(im, ax=axes[1], label='0=empty  1=P1  2=P2')

plt.tight_layout()
plt.show()


### 5.4  Decision Tree Training (ID3 / Information Gain)

We train a `DecisionTreeClassifier` with `criterion='entropy'` — this is the ID3 criterion (information gain). Because the raw board flattened to 43 integer features is already categorical-compatible, no further encoding is needed.

We also tune `max_depth` via cross-validation to balance accuracy vs. over-fitting, and explore the trade-off between tree depth and move quality.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle, warnings
warnings.filterwarnings('ignore')

# ── Train / test split (stratified by action) ────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

# ── Depth search via 5-fold CV ────────────────────────────────────────────
depths    = [3, 5, 8, 12, 18, 25, None]
cv_scores = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nDepth  |  CV Accuracy (mean ± std)")
print("-" * 38)
for d in depths:
    dt = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=skf, scoring='accuracy')
    cv_scores.append(scores.mean())
    label = str(d) if d else 'None'
    print(f"  {label:>4s}   |  {scores.mean():.4f} ± {scores.std():.4f}")

best_depth = depths[int(np.argmax(cv_scores))]
print(f"\nBest depth: {best_depth}")

# ── Final model at best depth ─────────────────────────────────────────────
dt_id3 = DecisionTreeClassifier(
    criterion='entropy', max_depth=best_depth, random_state=42
)
dt_id3.fit(X_train, y_train)

y_pred = dt_id3.predict(X_test)
print(f"\nTest accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Tree nodes    : {dt_id3.tree_.node_count}")
print(f"Tree leaves   : {dt_id3.get_n_leaves()}")


In [ ]:
# ── Depth vs accuracy plot ───────────────────────────────────────────────
depth_labels  = [str(d) if d else 'None' for d in depths]
train_accs = []
test_accs  = []

for d in depths:
    dt_tmp = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    dt_tmp.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_tmp.predict(X_train)))
    test_accs.append (accuracy_score(y_test,  dt_tmp.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))
x_ticks = range(len(depths))
ax.plot(x_ticks, train_accs, 'o-', label='Train accuracy', color='steelblue')
ax.plot(x_ticks, test_accs,  's--', label='Test accuracy',  color='tomato')
ax.plot(x_ticks, cv_scores,  '^:', label='CV accuracy',     color='seagreen')
ax.axvline(x=int(np.argmax(cv_scores)), color='gray', linestyle=':', alpha=.7,
           label=f'Best depth = {best_depth}')
ax.set_xticks(x_ticks)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('ID3 Decision Tree — Depth vs Accuracy')
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Classification report ────────────────────────────────────────────────
action_labels = [str(decode_move(i)) for i in range(N_ACTIONS)]
# Only report on classes present in test set
present = sorted(set(y_test))
print("Classification Report (actions present in test set):")
print(classification_report(
    y_test, y_pred,
    labels=present,
    target_names=[action_labels[i] for i in present],
    zero_division=0
))

# ── Confusion matrix heatmap ─────────────────────────────────────────────
import seaborn as sns
cm = confusion_matrix(y_test, y_pred, labels=present)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[action_labels[i] for i in present],
    yticklabels=[action_labels[i] for i in present],
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ID3 Decision Tree')
plt.tight_layout()
plt.show()


### 5.5  Decision Tree Visualisation

In [ ]:
# Top-level text view of the tree (first 4 levels)
print(export_text(dt_id3, feature_names=FEATURE_NAMES, max_depth=4))

# Graphical plot of the shallow subtree (max_depth=3 for readability)
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(
    dt_id3, max_depth=3,
    feature_names=FEATURE_NAMES,
    class_names=[str(decode_move(i)) for i in range(N_ACTIONS)],
    filled=True, rounded=True,
    impurity=True, proportion=False,
    ax=ax, fontsize=7
)
ax.set_title('ID3 Decision Tree (top 3 levels)')
plt.tight_layout()
plt.savefig('dt_id3_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tree visualisation saved to dt_id3_tree.png")


In [ ]:
# ── Top feature importances (ID3 information-gain based) ─────────────────
importances = pd.Series(dt_id3.feature_importances_, index=FEATURE_NAMES)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(9, 5))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Information-gain importance')
ax.set_title('Top 20 Most Important Features (ID3 Decision Tree)')
ax.grid(axis='x', alpha=.3)
plt.tight_layout()
plt.show()

print("Top 10 features:")
for fname, imp in top20.head(10).items():
    print(f"  {fname:12s}  {imp:.5f}")


### 5.6  DecisionTreeAgent — Playing with the Trained Tree

Wrap the ID3 tree in the same `choose_move(game)` interface as the MCTS agents so it can be dropped directly into `run_game` / `run_tournament`. The tree predicts the *best action label*; if that action happens to be illegal in the current state (the tree may predict a move that is blocked), we fall back to the highest-confidence **legal** action from the predicted probability distribution.

In [ ]:
class DecisionTreeAgent:
    """
    Game agent backed by a trained sklearn DecisionTreeClassifier.

    choose_move() mirrors the MCTS agent interface so it can be used
    in run_game() / run_tournament() without modification.

    Fallback strategy: if the tree's top-1 predicted action is illegal,
    we rank all 14 actions by predicted probability and return the
    highest-probability one that is currently legal.
    """

    def __init__(self, model: DecisionTreeClassifier):
        self.model = model
        self._classes = list(model.classes_)   # action labels seen during training

    def choose_move(self, game: PopOutGame) -> Optional[Tuple[str, int]]:
        legal = game.get_legal_moves()
        if not legal:
            return None

        features = np.array([state_to_features(game)], dtype=np.int8)
        proba    = self.model.predict_proba(features)[0]   # shape (n_classes,)

        # Build a full probability array indexed 0-13
        full_proba = np.zeros(N_ACTIONS, dtype=float)
        for cls_idx, cls_label in enumerate(self._classes):
            full_proba[cls_label] = proba[cls_idx]

        # Rank legal actions by predicted probability
        legal_encoded = {encode_move(m): m for m in legal}
        best_label = max(legal_encoded.keys(),
                         key=lambda lbl: full_proba[lbl])
        return legal_encoded[best_label]


dt_agent = DecisionTreeAgent(dt_id3)

# Quick sanity check — one full game DT vs DT
game_check = PopOutGame()
while not game_check.game_over:
    m = dt_agent.choose_move(game_check)
    if m is None: break
    game_check.make_move(*m)
print("DT self-play check:", game_check.get_status())
print(f"Game lasted {len(game_check.move_history)} moves")


In [ ]:
# ── Persist the trained ID3 model ────────────────────────────────────────
import pickle

DT_MODEL_PATH = 'dt_id3_model.pkl'
with open(DT_MODEL_PATH, 'wb') as fh:
    pickle.dump(dt_id3, fh)
print(f"ID3 model saved → {DT_MODEL_PATH}")

# Reload test
with open(DT_MODEL_PATH, 'rb') as fh:
    dt_id3_loaded = pickle.load(fh)
assert accuracy_score(y_test, dt_id3_loaded.predict(X_test)) == accuracy_score(y_test, y_pred)
print("Reload verification: OK")


## 6. Visualize Game Trees and Results

In [ ]:
# ── Win-rate heatmap across all matchups ─────────────────────────────────
agent_names = list(strategies.keys())
n = len(agent_names)
win_matrix = pd.DataFrame(np.full((n, n), np.nan),
                           index=agent_names, columns=agent_names)

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    win_matrix.loc[a1, a2] = row['A1_win%']
    win_matrix.loc[a2, a1] = row['A2_win%']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win-rate heatmap
sns.heatmap(win_matrix.astype(float), annot=True, fmt='.1f',
            cmap='RdYlGn', vmin=0, vmax=100, linewidths=.5,
            ax=axes[0], cbar_kws={'label': 'Win %'})
axes[0].set_title('Win % (row agent vs column agent)')

# Average move time comparison
time_data = {}
for _, row in df_tournament.iterrows():
    for ag, col in [(row['A1'], 'A1_ms'), (row['A2'], 'A2_ms')]:
        time_data.setdefault(ag, []).append(row[col])
avg_times = {ag: np.mean(v) for ag, v in time_data.items()}
pd.Series(avg_times).sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Average ms per move')
axes[1].set_title('Avg Move Time per Agent')
axes[1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.show()

# ── Overall win-rate per agent (sum across all matchups) ──────────────────
print("\nOverall performance summary:")
total_wins = {ag: 0 for ag in agent_names}
total_games= {ag: 0 for ag in agent_names}
for _, row in df_tournament.iterrows():
    for ag, wcol, gcol in [
        (row['A1'], 'A1_wins', 'A1_wins'),
        (row['A2'], 'A2_wins', 'A2_wins'),
    ]:
        total_wins[ag]  += row[wcol]
        total_games[ag] += row['A1_wins'] + row['A2_wins'] + row['Draws']
for ag in agent_names:
    g = total_games[ag]
    w = total_wins[ag]
    print(f"  {ag:12s}  {w:3d} wins / {g:3d} games  ({w/g*100:.1f}%)")


## 7. Summary and Conclusions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 7.  Identify & Persist the Best MCTS Agent
# ══════════════════════════════════════════════════════════════════════════

# ── Identify the best MCTS strategy from tournament results ──────────────
mcts_names = ['Standard', 'RAVE', 'Top-K', 'Heuristic']
mcts_scores = {ag: 0 for ag in mcts_names}

for _, row in df_tournament.iterrows():
    a1, a2 = row['A1'], row['A2']
    # Only count MCTS vs MCTS matchups for the ranking
    if a1 in mcts_names and a2 in mcts_names:
        mcts_scores[a1] += row['A1_wins']
        mcts_scores[a2] += row['A2_wins']

best_mcts_name = max(mcts_scores, key=mcts_scores.get)
print("MCTS win counts (inter-MCTS games only):")
for ag, sc in sorted(mcts_scores.items(), key=lambda x: -x[1]):
    print(f"  {ag:12s}  {sc} wins")
print(f"\nBest MCTS strategy: {best_mcts_name}")

# ── Build the production-strength best agent ──────────────────────────────
# Use more iterations than tournament (better quality for actual gameplay)
PRODUCTION_ITER = 500
best_mcts_map = {
    'Standard'  : MCTS(iterations=PRODUCTION_ITER),
    'RAVE'      : MCTS_RAVE(iterations=PRODUCTION_ITER),
    'Top-K'     : MCTSTopK(iterations=PRODUCTION_ITER, k=7),
    'Heuristic' : MCTSWithHeuristics(iterations=PRODUCTION_ITER),
}
best_mcts_agent = best_mcts_map[best_mcts_name]

# ── Persist with pickle ───────────────────────────────────────────────────
import pickle

BEST_MCTS_PATH = 'best_mcts_agent.pkl'
with open(BEST_MCTS_PATH, 'wb') as fh:
    pickle.dump(best_mcts_agent, fh)
print(f"Best MCTS agent saved → {BEST_MCTS_PATH}")

# ── Reload & verify ───────────────────────────────────────────────────────
with open(BEST_MCTS_PATH, 'rb') as fh:
    best_mcts_loaded = pickle.load(fh)

game_verify = PopOutGame()
game_verify.make_move('drop', 3)
test_move = best_mcts_loaded.choose_move(game_verify)
print(f"Reload verification — suggested move: {test_move}  ✓")

print()
print("=" * 55)
print(f" Best MCTS : {best_mcts_name} ({PRODUCTION_ITER} iterations)")
print(f" Saved to  : {BEST_MCTS_PATH}")
print(f" ID3 Tree  : dt_id3_model.pkl")
print("=" * 55)
print()
print("Usage in interface.py / game loop:")
print("  import pickle")
print(f"  with open('{BEST_MCTS_PATH}', 'rb') as f:")
print("      agent = pickle.load(f)")
print("  move = agent.choose_move(game)")


## 8. API Execution/Visualization (Best MCTS vs ID3)